In [1]:
from sympy.physics.units import temperature

from avito.constants import SPLIT_MARKERS
from avito.prompts import shouldSplitInstructionPrompt, shouldSplitPromptWithRAG
from common.mistral import call_mistral, MistralCallConfig
from common.logger import AVITO_SHOULD_SPLIT_LOGGER as logger

from langgraph.graph import StateGraph
from langgraph.graph import START, END
from typing import *

import re

from pydantic import BaseModel

from common.paths import get_avito_data_dpath
import pandas as pd
import ast

In [2]:
data_dpath = get_avito_data_dpath()

In [3]:
mc_map_fpath = data_dpath / "rnc_mic_key_phrases.csv"
mc_map_df = pd.read_csv(mc_map_fpath)

mc_map = {
    row["mcId"]: row["mcTitle"]
    for _, row in mc_map_df.iterrows()
}
keyphrases = [string.split("; ") for string in mc_map_df["keyPhrases"].tolist()]

print(keyphrases[:5])
print(mc_map)
mc_map_df.head()

[['ремонт под ключ', 'комплексный ремонт', 'полный ремонт квартиры', 'ремонт квартиры под ключ', 'ремонт дома под ключ', 'ремонт коттеджа под ключ', 'капитальный ремонт под ключ', 'косметический ремонт под ключ', 'комплексная отделка', 'полная отделка квартиры', 'ремонт помещений под ключ', 'весь цикл ремонта', 'ремонт от демонтажа до чистовой', 'ремонт с нуля под ключ', 'ремонт вторички под ключ', 'ремонт новостройки под ключ', 'ремонт офиса под ключ', 'ремонт комнаты под ключ', 'ремонт кухни под ключ', 'ремонт ванной под ключ', 'комплекс работ по ремонту', 'ремонт с материалами', 'ремонт без посредников под ключ', 'ремонт с дизайн проектом', 'ремонт с черновой и чистовой отделкой', 'сдаем объект под ключ', 'ведем ремонт полностью', 'выполняем весь комплекс работ', 'берем объект целиком', 'ремонт под сдачу', 'ремонт для проживания под ключ', 'отделка под ключ', 'полное обновление квартиры', 'ремонт жилья под ключ', 'комплексный ремонт квартиры', 'комплексный ремонт дома', 'ремонт под 

,mcId,mcTitle,keyPhrases,description
0,101,Ремонт квартир и домов под ключ,ремонт под ключ; комплексный ремонт; полный ре...,Комплексный ремонт объекта целиком
1,102,Сантехника,сантехника; сантехнические работы; услуги сант...,Сантехнические работы
2,103,Электрика,электрика; электромонтажные работы; услуги эле...,Электромонтажные работы
3,104,Натяжные потолки,натяжные потолки; монтаж натяжных потолков; ус...,Монтаж и обслуживание натяжных потолков
4,105,Укладка плитки,укладка плитки; плиточные работы; плиточник; у...,Плиточные работы


In [4]:
data_fpath = data_dpath / "rnc_dataset_markup.json"
data_df = pd.read_json(data_fpath)
data_df["targetSplitMcIds"] = data_df["targetSplitMcIds"].apply(ast.literal_eval)
data_df["targetSplitMcTitles"] = data_df["targetSplitMcIds"].apply(lambda mc_ids: [mc_map[mc_id] for mc_id in mc_ids])



def clean_description(text: str) -> str:
    # Оставляем только кириллицу и обычные пробелы
    text = re.sub(r'[^а-яА-ЯёЁ ]', ' ', text)
    # Схлопываем множественные пробелы
    text = re.sub(r' +', ' ', text)
    return text.strip()

data_df["description"] = data_df["description"].apply(clean_description)

from sklearn.model_selection import train_test_split
data_df, val_df = train_test_split(data_df, test_size=0.2, random_state=42)

data_df.head()

,itemId,sourceMcId,sourceMcTitle,description,targetDetectedMcIds,targetSplitMcIds,shouldSplit,caseType,split,targetSplitMcTitles
1125,1001139,101,Ремонт квартир и домов под ключ,Отделка под ключ косметический ремонт помощь в...,[],[],False,no_other_microcategories_detected,NaN,[]
1767,1001787,101,Ремонт квартир и домов под ключ,Уважаемые дамы и господа предлагаем услуги рем...,"[102, 103, 104, 108]",[],False,other_microcategories_detected_but_not_split,NaN,[]
756,1000769,101,Ремонт квартир и домов под ключ,Произведем монтаж натяжного потолка любой слож...,[104],[104],True,split,NaN,[Натяжные потолки]
2471,1002492,101,Ремонт квартир и домов под ключ,Все виды внутри отделочных работ,[],[],False,no_other_microcategories_detected,NaN,[]
1565,1001585,101,Ремонт квартир и домов под ключ,Хотите обновить интерьер своей квартиры но не ...,"[102, 103, 104, 111]",[],False,other_microcategories_detected_but_not_split,NaN,[]


In [10]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings

embedding_model = SentenceTransformerEmbeddings(model_name="sergeyzh/rubert-mini-frida")
try:
        vectorstore = Chroma(
        embedding_function=embedding_model,
        collection_name="avito_descriptions",
    )
except Exception as e:
    logger.warning(f"Не удалось загрузить существующий векторный стор: {e}")
    vectorstore = Chroma.from_texts(
        texts=data_df["description"].tolist(),
        embedding=embedding_model,
        collection_name="avito_descriptions",
    )
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# save vectorstore
vectorstore.persist()

def get_balanced_top_k(desc: str, k: int = 6) -> list:
    results = retriever.invoke(desc, k=20)
    finded_texts = [r.page_content for r in results if r.page_content != desc]

    true_examples = [
        t for t in finded_texts
        if data_df.loc[data_df["description"] == t, "shouldSplit"].values[0]
    ]
    false_examples = [
        t for t in finded_texts
        if not data_df.loc[data_df["description"] == t, "shouldSplit"].values[0]
    ]

    # берём поровну
    half = k // 2
    balanced = true_examples[:half] + false_examples[:half]

    result = []
    for text in balanced:
        cats = data_df.loc[data_df["description"] == text, "targetSplitMcTitles"].values[0]
        verdict = data_df.loc[data_df["description"] == text, "shouldSplit"].values[0]
        result.append((text, verdict, cats))

    return result



Default prompt name is set to 'Classification'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


In [11]:
# utils, nodes, States


def keywords_pre_filter(state: RunState) -> RunState:
    desc = state.description

    split_words_count = 0
    for word in SPLIT_MARKERS:
        if word in desc:
            split_words_count += 1
    split_markers_flag = split_words_count > 0

    # keyphrases
    keyphrases_vec = [0] * len(keyphrases)
    for idx, keyphrase_list in enumerate(keyphrases):
        for keyphrase in keyphrase_list:
            if keyphrase in desc:
                keyphrases_vec[idx] = 1
                break

    if sum(keyphrases_vec) > 1:
        keyphrases_flag = True
    else:
        keyphrases_flag = False

    if split_markers_flag or keyphrases_flag:
        state.shouldSplit = True
    else:
        state.shouldSplit = False

    return state

# example
str_ex = """
бригада мастеров со стажем более 20 лет выполним внутренние отделки домов, квартир офисы И. Т. Д,работа под ключ или же как вам удобно, Шпаклëвка под обои, под покраску,покраска, Штукатурка ,падуги гипсовые, пластиковые, обои,кафель, отопления,ламинат,И.Т.Д, также выполним фасадные работы, утепление пеноплексом, облицовку природным камнем, И. Т. Д. работаем по местным ценам, делаем скидку на каждый вид работы индивидуальный подход к каждому виду работы ,качество и своевременное сдача объекта гарантируем если не возникнет внеплановая ситуация, помощь в подборе материалов вместе с заказчиком, бригада без вредных привычек,работаем полный  рабочий день от выходного до выходного,инструментами обеспечены полностью,пишите. Звоните.
"""
state = RunState(description=str_ex)
state.description = clean_description(state.description)
state = keywords_pre_filter(state)

retriever_results = retriever.invoke(state.description)

finded_texts = [item.page_content for item in retriever_results]
finded_categories = [data_df.loc[data_df["description"] == text, "targetSplitMcTitles"].values[0] for text in finded_texts]
finded_verdicts = [data_df.loc[data_df["description"] == text, "shouldSplit"].values[0] for text in finded_texts]

top_k = [(text, verdict, categories) for text, verdict, categories in zip(finded_texts, finded_verdicts, finded_categories)]

prompt = shouldSplitPromptWithRAG(
    desc=state.description,
    top_k=top_k
)

print(prompt)

Похожие объявления из базы:


Объявление для классификации:
бригада мастеров со стажем более лет выполним внутренние отделки домов квартир офисы И Т Д работа под ключ или же как вам удобно Шпакл вка под обои под покраску покраска Штукатурка падуги гипсовые пластиковые обои кафель отопления ламинат И Т Д также выполним фасадные работы утепление пеноплексом облицовку природным камнем И Т Д работаем по местным ценам делаем скидку на каждый вид работы индивидуальный подход к каждому виду работы качество и своевременное сдача объекта гарантируем если не возникнет внеплановая ситуация помощь в подборе материалов вместе с заказчиком бригада без вредных привычек работаем полный рабочий день от выходного до выходного инструментами обеспечены полностью пишите Звоните

Нужно ли выделять отдельные микрокатегории?


In [7]:
mistral_config = MistralCallConfig(
     models_list=["mistral-medium-latest"]
)
response = call_mistral(
    mistral_config,
    messages= [
        {
            "role": "system",
            "content": shouldSplitInstructionPrompt
        },
        {
            "role": "user",
            "content": prompt
        },
    ],
    reasoning_effort="hard"

)
print(response)

2026-04-08 16:57:17,696 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-04-08 16:57:17,696 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'reasoning_effort']
2026-04-08 16:57:17,696 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью mistral-medium-latest, ключ 1/21
2026-04-08 16:57:17,885 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-04-08 16:57:17,885 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-04-08 16:57:17,895 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели mistral-medium-latest
2026-04-08 16:57:18,414 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: получен ответ длиной 5 символов
2026-04-08 16:57:18,416 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: успешное выполнение на попытке 1
2026-04-08 16:57:18,417 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: успешный вызов модели mistral-medium-latest


False


In [13]:
from tqdm.auto import tqdm

subsample_df = val_df.sample(50, random_state=43)
preds = []
for _, row in tqdm(subsample_df.iterrows(), total=len(subsample_df)):
    state = RunState(description=row["description"])
    state.description = clean_description(state.description)
    state = keywords_pre_filter(state)

    retriever_results = retriever.invoke(state.description)

    finded_texts = [item.page_content for item in retriever_results]
    finded_texts_without_gt = [text for text in finded_texts if text != state.description]

    finded_categories = [data_df.loc[data_df["description"] == text, "targetSplitMcTitles"].values[0] for text in finded_texts]
    finded_verdicts = [data_df.loc[data_df["description"] == text, "shouldSplit"].values[0] for text in finded_texts]

    top_k = [(text, verdict, categories) for text, verdict, categories in zip(finded_texts, finded_verdicts, finded_categories)]

    prompt = shouldSplitPromptWithRAG(
        desc=state.description,
        top_k=top_k
    )

    mistral_config = MistralCallConfig(
         models_list=["mistral-medium-latest"]
    )
    response = call_mistral(
        mistral_config,
        messages= [
            {
                "role": "system",
                "content": shouldSplitInstructionPrompt
            },
            {
                "role": "user",
                "content": prompt
            },
        ],
        reasoning_effort="hard"

    )
    preds.append("True" in response)

subsample_df["mistral_rag_prediction"] = preds

print(f"Accuracy: {(subsample_df['shouldSplit'] == subsample_df['mistral_rag_prediction']).mean():.4f}")
subsample_df[["description", "shouldSplit", "mistral_rag_prediction"]] # noqa

  0%|          | 0/50 [00:00<?, ?it/s]

2026-04-08 16:44:07,536 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-04-08 16:44:07,538 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'reasoning_effort']
2026-04-08 16:44:07,542 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью mistral-medium-latest, ключ 3/21
2026-04-08 16:44:07,574 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-04-08 16:44:07,576 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-04-08 16:44:07,580 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели mistral-medium-latest
2026-04-08 16:44:19,793 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: получен ответ длиной 4 символов
2026-04-08 16:44:19,796 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: успешное выполнение на попытке 1
2026-04-08 16:44:19,797 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: успешный вызов модели mistral-medium-latest
2026-04-08 16:44:19,825 - m

Accuracy: 0.5400


,description,shouldSplit,mistral_rag_prediction
1025,Ремонт квартир домов любой сложности Монтаж де...,False,True
1888,бригада строителей выполняем все виды ремонта ...,False,True
1005,Отделка домов и квартир Штукатурка Шпаклевка П...,False,True
2397,Здравствуйте Меня зовут Александр я инженер ст...,False,True
210,Делаю все виды отделочных работ,False,False
1362,Плитка обои ламинат и прочее Где надо оштукату...,False,True
1034,Предоставляю услуги мастера плотника универсал...,False,True
1709,Самая низкая цена по Краснодару и Краснодарско...,False,False
1956,Мы небольшая частная бригада по ремонту кварти...,False,True
1790,Ремонт квартир и отделка любой сложности косме...,False,False


In [18]:
class RunState(BaseModel):
    description: str
    shouldSplit: bool = False
    embedding: Optional[list[float]] = None
    top_k_examples: Optional[list[tuple[str, bool, list[str]]]] = None
    drafts: list[str] = []

    last_node: Optional[str] = None
    metadata: dict[Any, Any] = {}

from dataclasses import dataclass

class ShouldSplitGraph:
    def __init__(self):
        self.verbose = True

        @dataclass
        class nodes:
            clean = "clean"
            pre_filter = "pre_filter"
            pre_filter_exit = "pre_filter_exit"
            rag_split = "rag_split"
            llm_exit = "llm_exit"
            drafts = "drafts"
            drafts_exit = "drafts_exit"

        self.nodes = nodes

        builder = StateGraph(RunState)

        builder.add_node("clean", self.clean_description_node)
        builder.add_node("pre_filter", self.keywords_pre_filter_logged)
        builder.add_node("pre_filter_exit", self.pre_filter_exit)

        builder.add_node("rag_split", self.rag_should_split)
        builder.add_node("llm_exit", self.llm_exit)

        builder.add_node("drafts", self.generate_drafts)
        builder.add_node("drafts_exit", self.drafts_exit)

        builder.set_entry_point("clean")
        builder.add_edge("clean", "pre_filter")

        builder.add_conditional_edges(
            "pre_filter",
            lambda state: "rag_split" if state.shouldSplit else "pre_filter_exit"
        )

        builder.add_conditional_edges(
            "rag_split",
            self.should_split_router,
            {"split": "drafts", END: "llm_exit"}
        )

        builder.add_edge("drafts", "drafts_exit")

        builder.add_edge("pre_filter_exit", END)
        builder.add_edge("llm_exit", END)
        builder.add_edge("drafts_exit", END)

        self.graph = builder.compile()

    def _log(self, msg: str):
        if self.verbose:
            logger.info(msg)

    def clean_description_node(self, state: RunState) -> RunState:
        before = len(state.description)
        state.description = clean_description(state.description)
        after = len(state.description)

        self._log("━" * 50)
        self._log(f"🧹 [clean] {before} → {after} символов")
        return state

    def keywords_pre_filter_logged(self, state: RunState) -> RunState:
        state = keywords_pre_filter(state)

        self._log("━" * 50)
        self._log(f"🔑 [pre_filter] shouldSplit={state.shouldSplit}")
        if not state.shouldSplit:
            self._log("   Ключевые фразы/маркеры не найдены → завершаем на pre_filter")
        else:
            self._log("   Префильтр сработал → идём в RAG/LLM")
        return state

    def pre_filter_exit(self, state: RunState) -> RunState:
        self._log("━" * 50)
        self._log("⛔ [pre_filter_exit] Пайплайн завершён после префильтра")
        return state

    def rag_should_split(self, state: RunState) -> RunState:
        self._log("━" * 50)
        self._log("🔍 [rag_split] СТАРТ")
        self._log(f"   Текст ({len(state.description)} символов): {state.description[:100]}...")

        retriever_results = get_balanced_top_k(state.description, k=6)
        finded_texts = [item.page_content for item in retriever_results]
        finded_texts = [text for text in finded_texts if text != state.description]
        self._log(f"   RAG вернул {len(finded_texts)} кандидатов (без точного совпадения)")

        finded_categories = [
            data_df.loc[data_df["description"] == text, "targetSplitMcTitles"].values[0]
            for text in finded_texts
        ]
        finded_verdicts = [
            data_df.loc[data_df["description"] == text, "shouldSplit"].values[0]
            for text in finded_texts
        ]

        top_k = list(zip(finded_texts, finded_verdicts, finded_categories))

        self._log("   Топ-k примеры:")
        for i, (text, verdict, cats) in enumerate(top_k):
            self._log(f"   [{i+1}] shouldSplit={verdict} | кат={cats} | текст: {text[:60]}...")

        prompt = shouldSplitPromptWithRAG(desc=state.description, top_k=top_k)

        self._log("   Отправляем запрос в Mistral...")
        response = call_mistral(
            MistralCallConfig(models_list=["mistral-medium-latest"]),
            messages=[
                {"role": "system", "content": shouldSplitInstructionPrompt},
                {"role": "user", "content": prompt},
            ],
            reasoning_effort="hard",
            temperature=0.0,
        )

        decision = "True" in str(response)
        self._log(f"   Ответ модели: {str(response)[:100]}")
        self._log(f"   Решение: {'✅ SPLIT' if decision else '❌ NO SPLIT'}")

        state.shouldSplit = decision
        return state

    def should_split_router(self, state: RunState) -> str:
        route = "split" if state.shouldSplit else END
        self._log(f"🔀 [router] → {route}")
        return route

    def llm_exit(self, state: RunState) -> RunState:
        self._log("━" * 50)
        self._log("⛔ [llm_exit] Пайплайн завершён после LLM")
        self._log(f"   shouldSplit={state.shouldSplit}")
        return state

    def generate_drafts(self, state: RunState) -> RunState:
        self._log("━" * 50)
        self._log("📝 [drafts] СТАРТ — генерация черновиков")
        self._log("📝 [drafts] КОНЕЦ")
        return state

    def drafts_exit(self, state: RunState) -> RunState:
        self._log("━" * 50)
        self._log("✅ [drafts_exit] Пайплайн завершён после drafts")
        return state

    def invoke(self, description: str, verbose: bool = True) -> RunState:
        prev_verbose = self.verbose
        self.verbose = verbose

        try:
            self._log("═" * 50)
            self._log("🚀 [pipeline] СТАРТ")
            self._log(f"   Вход: {description[:100]}...")

            state = RunState(description=description)
            result = self.graph.invoke(state)
            final = RunState(**result)

            self._log("═" * 50)
            self._log(f"🏁 [pipeline] КОНЕЦ | shouldSplit={final.shouldSplit}")
            self._log("═" * 50)

            return final
        finally:
            self.verbose = prev_verbose

pipeline = ShouldSplitGraph()
result = pipeline.invoke(str_ex, verbose=True)
print(result.shouldSplit)

2026-04-08 17:19:02,785 - avito-should-split - INFO - [SHOULD_SPLIT] ══════════════════════════════════════════════════
2026-04-08 17:19:02,787 - avito-should-split - INFO - [SHOULD_SPLIT] 🚀 [pipeline] СТАРТ
2026-04-08 17:19:02,789 - avito-should-split - INFO - [SHOULD_SPLIT]    Вход: 
бригада мастеров со стажем более 20 лет выполним внутренние отделки домов, квартир офисы И. Т. Д,ра...
2026-04-08 17:19:02,794 - avito-should-split - INFO - [SHOULD_SPLIT] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026-04-08 17:19:02,797 - avito-should-split - INFO - [SHOULD_SPLIT] 🧹 [clean] 736 → 707 символов
2026-04-08 17:19:02,800 - avito-should-split - INFO - [SHOULD_SPLIT] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026-04-08 17:19:02,802 - avito-should-split - INFO - [SHOULD_SPLIT] 🔑 [pre_filter] shouldSplit=False
2026-04-08 17:19:02,805 - avito-should-split - INFO - [SHOULD_SPLIT]    Ключевые фразы/маркеры не найдены → завершаем на pre_filter
2026-04-08 17:19:02,808 - avito-should

False


In [13]:
from tqdm.auto import tqdm
val_scores = []
subsample_df = val_df.sample(200, random_state=12)
for _, row in tqdm(subsample_df.iterrows(), total=len(subsample_df)):
    result = pipeline.invoke(row["description"], verbose=False)
    val_scores.append(result.shouldSplit)

subsample_df["pipeline_prediction"] = val_scores
print(f"Pipeline Accuracy: {(subsample_df['shouldSplit'] == subsample_df['pipeline_prediction']).mean():.4f}")
subsample_df[["description", "shouldSplit", "pipeline_prediction"]] # noqa

  0%|          | 0/200 [00:00<?, ?it/s]

2026-04-08 17:09:04,585 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-04-08 17:09:04,588 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'reasoning_effort', 'temperature']
2026-04-08 17:09:04,591 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью mistral-medium-latest, ключ 1/21
2026-04-08 17:09:04,784 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-04-08 17:09:04,784 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-04-08 17:09:04,791 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели mistral-medium-latest
2026-04-08 17:09:16,565 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: получен ответ длиной 5 символов
2026-04-08 17:09:16,567 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: успешное выполнение на попытке 1
2026-04-08 17:09:16,569 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: успешный вызов модели mistral-medium-latest
2026-04-08 1

Pipeline Accuracy: 0.7700


,description,shouldSplit,pipeline_prediction
1377,Ванные комнаты и сан узлы под ключ Демонтаж пе...,True,False
2054,ремонт квартир качественно плиточные работы и ...,False,False
2084,Профес ион льно недорог и кач ств нн п оизв ду...,False,False
2279,Любые ремонтные работы а так же работы по демо...,False,True
1632,Ремонт любой сложности Качество и гарантия сде...,False,False
...,...,...,...
199,Весь спектр отделочных работ штукатурка шпакле...,True,False
1089,делаем капитальный внутренний ремонт а также м...,True,True
2314,Предлагаю Вам услугу в сфере ремонта квартир д...,True,True
1769,строительные работы,False,False


In [14]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print(classification_report(subsample_df["shouldSplit"], subsample_df["pipeline_prediction"]))
print("Confusion Matrix:")
print(confusion_matrix(subsample_df["shouldSplit"], subsample_df["pipeline_prediction"]))
print(f"F1 Score: {f1_score(subsample_df['shouldSplit'], subsample_df['pipeline_prediction']):.4f}")

              precision    recall  f1-score   support

       False       0.81      0.93      0.87       159
        True       0.35      0.15      0.21        41

    accuracy                           0.77       200
   macro avg       0.58      0.54      0.54       200
weighted avg       0.72      0.77      0.73       200

Confusion Matrix:
[[148  11]
 [ 35   6]]
F1 Score: 0.2069


In [15]:
# some fp samples

fp_samples = subsample_df[(subsample_df["shouldSplit"] == False) & (subsample_df["pipeline_prediction"] == True)]
for idx, row in fp_samples.iterrows():
    print(f"Описание: {row['description']}")
    print(f"shouldSplit: {row['shouldSplit']} | pipeline_prediction: {row['pipeline_prediction']}")
    print("-" * 80)

Описание: Любые ремонтные работы а так же работы по демонтажу Выравнивание стен Поклейка обоев Покраска Работа с гипсокартоном Укладка плитки Обшивка панелями Возведение стен Стяжка пола Укладка линолеум ламинат кварц винил Установка окон остекление балконов Установка дверей Шпатлевка стен и потолков Сборка разборка мебели Любые Сантехнические работы Возможен выезд за Город Пишите Звоните Также работаем с Юр лицами
shouldSplit: False | pipeline_prediction: True
--------------------------------------------------------------------------------
Описание: Здравствуйте я Татьяна Работаю сама или с бригадой Мы качественно сделаем косметический ремонт выполним подготовку поклеим обои без швов и пузырей Свою работу любим выполняем все аккуратно с душой дляВас Работаем срочно день в день и по записи Имеем много постоянных клиентов для них вне очереди Работаем без выходных за собой оставляем чистоту и порядок Предоплату не берем Выезд на замер и консультация Бесплатно Варианты оплаты Наличными Пе

In [17]:
positive_samples = subsample_df.loc[subsample_df["shouldSplit"]==True].sample(5)
for idx, row in positive_samples.iterrows():
    print(f"Описание: {row['description']}")
    print(f"shouldSplit: {row['shouldSplit']} | pipeline_prediction: {row['pipeline_prediction']}")
    print("-" * 80)

Описание: БЕЗ ПОСРЕДНИКОВ МАСТЕР строительно монтажных и отделочных работ Ремонт одной рукой КАЧЕСТВО СРОКИ УДАЛЕННО ДЕЛАЮ ВСЁ Звоните всё обсудим и воплотим Фото в профиле моих работ видео по запросу Делаю абсолютно все кроме натяжного потолка и установки кондиционера Современное оборудование и инструмент имеется Скидки на стройматериалы имею в большинстве магазинов Доставлю и закуплю материалы сам предоставлю полный финансовый отчёт ЗВОНИТЕ прямо сейчас сдал очередной объект готов брать новый поэтому объявление актуально Ремонт Новостройка Косметический Эконом Стандарт Капитальный Евроремонт По дизайн проекту или картинке с ваших слов и тд Электрика монтаж проводки установка розеток выключателей люстр освещения подсветки замена проводки Выравнивание Штукатурка стяжка шпатлевка стен и потолка откосы шпатлевка под покраску и обои Стены и пол монтаж перегородок шумоизоляция гипсокартон покраска стен обои дерево вагонка пластик плитка кафель укладка ламината керамогранит кварцвинил парке

In [10]:
from common.paths import get_avito_gitignore_data_dpath
import pandas as pd

annoted_fpath = get_avito_gitignore_data_dpath() / "20260408_191949.json"
annoted_df = pd.read_json(annoted_fpath)

mc_map_fpath = data_dpath / "rnc_mic_key_phrases.csv"
mc_map_df = pd.read_csv(mc_map_fpath)

mc_map = {
    row["mcId"]: row["mcTitle"]
    for _, row in mc_map_df.iterrows()
}

import ast
annoted_df["targetSplitMcIds"] = annoted_df["targetSplitMcIds"].apply(ast.literal_eval)
annoted_df["targetSplitMcTitles"] = annoted_df["targetSplitMcIds"].apply(lambda mc_ids: [mc_map[mc_id] for mc_id in mc_ids])

annoted_df.head()

,itemId,sourceMcId,sourceMcTitle,description,targetDetectedMcIds,targetSplitMcIds,shouldSplit,targetSplitMcTitles
0,1000001,101,Ремонт квартир и домов под ключ,"Всё виды строительных работ\r\nКачественно, в ...",[],[],False,[]
1,1000002,101,Ремонт квартир и домов под ключ,Профессионально и качественно сделаем ремонт к...,[],[],False,[]
2,1000003,101,Ремонт квартир и домов под ключ,"ремонт квартир, ванной комнате , балкон",[],[],False,[]
3,1000004,101,Ремонт квартир и домов под ключ,ЗBОНИТЕ KОHСУЛЬТАЦИЯ БЕCПЛАTНAЯ ПO ТУЛЬСKOЙ ОБ...,[],[],False,[]
4,1000005,101,Ремонт квартир и домов под ключ,Ремонт квартир любой сложности. Квартиры под к...,[],[],True,[]


In [11]:
from avito.should_split.retrieval import ChromaRetriever, VectorDbConfig, EncoderConfig

vdb_cfg = VectorDbConfig(
    collection_name="avito_descriptions",
    persist=True,
)

encoder_cfg = EncoderConfig(
    model_name="sergeyzh/rubert-mini-frida"
)

retriever = ChromaRetriever(
    examples_df=annoted_df,
    vector_db_config=vdb_cfg,
    encoder_config=encoder_cfg
)

Default prompt name is set to 'Classification'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


In [12]:
from avito.should_split.catalog import load_microcategory_catalog
from avito.should_split.config import ShouldSplitGraphConfig
from avito.should_split.pipeline import ShouldSplitPipeline
from common.paths import get_avito_data_dpath

cfg = ShouldSplitGraphConfig.from_default_yaml()

data_dpath = get_avito_data_dpath()

mc_map_fpath = data_dpath / cfg.data.mc_map_filename
markup_fpath = data_dpath / cfg.data.markup_filename

cfg.graph.use_rag = True
graph = ShouldSplitPipeline(
    catalog=load_microcategory_catalog(mc_map_fpath),
    retriever=retriever,
    config=cfg,
    examples_df=annoted_df
)

example_desc = annoted_df.sample(1, random_state=42)["description"].values[0]
result = graph.invoke(example_desc)
print(f"Final decision: shouldSplit={result}")

2026-04-08 20:32:50,964 - avito-should-split - INFO - [SHOULD_SPLIT] [clean] 1401 -> 1310 символов
2026-04-08 20:32:50,980 - avito-should-split - INFO - [SHOULD_SPLIT] [pre_filter] shouldSplit=True, split_markers=1, categories=0
2026-04-08 20:32:51,015 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-04-08 20:32:51,032 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'reasoning_effort', 'temperature']
2026-04-08 20:32:51,036 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью mistral-medium-latest, ключ 1/21
2026-04-08 20:32:51,055 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-04-08 20:32:51,055 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-04-08 20:32:51,055 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели mistral-medium-latest
2026-04-08 20:33:02,720 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: получен ответ длиной 5 символов
2026-04-

Final decision: shouldSplit=description='здравствуйте мы предлагаем комплексные услуги с нами ваш ремонт получится без забот выполняем ремонт квартир домов коммерческих помещений под ключ любой сложности наша локация западный обход жк самолет жк достояние жк парк победы жк архитектор бригада мастеров готовы приступить к ремонту в ближайшее время все с опытом работы выполняем любые виды отделочных и ремонтных работ от стандартных вариантов до эксклюзивного проекта работаем по дизайн проекту и без авторский надзор подберем все материалы и мебель от наших партнёров по самый выгодной цене привезём при необходимости из краснодара всегда заключаем договор фиксируем дату окончания ремонта и стоимость работ также работаем дистанционно у нас большой опыт и мы сможем организовать весь процесс ремонта от начала до конца ваши выгоды при работе с нами все отделочные работы выполняем точно в срок без задержек еженедельный фотоотчет по а арр закон о тишине и чистоту на объекте соблюдаем и поддерживае

False